In [1]:
import torch
import torch.nn as nn 
import os
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
import faiss
import numpy as np
import pickle
import h5py

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, 
                 product_descriptions, values, units, prices, processor, max_length=77):
        self.img_dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.prices = torch.tensor(prices, dtype=torch.float32)
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """
        Concatenate all text fields into one string.
        Prioritizes most important information first (for token limit).
        """
        text_parts = []
        
        # Item Name (most important)
        if pd.notna(self.item_names[idx]) and str(self.item_names[idx]).strip():
            text_parts.append(str(self.item_names[idx]).strip())
        
        # Quantity (concise and important for pricing)
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"{self.values[idx]} {self.units[idx]}")
        
        # Bullet Points (key features)
        if pd.notna(self.bullet_points[idx]) and str(self.bullet_points[idx]).strip():
            text_parts.append(str(self.bullet_points[idx]).strip())
        
        # Product Description (detailed, might be truncated)
        if pd.notna(self.product_descriptions[idx]) and str(self.product_descriptions[idx]).strip():
            text_parts.append(str(self.product_descriptions[idx]).strip())
        
        # Join with separator that CLIP handles well
        full_text = ". ".join(text_parts)
        
        # Handle empty text case
        if not full_text.strip():
            full_text = "product"  # Fallback text
        
        return full_text
    
    def __getitem__(self, idx):
        """
        Returns a single sample with image and text features.
        """
        # Load image
        img_filename = f"{self.image_paths[idx]}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Return a blank image as fallback
            img = Image.new('RGB', (224, 224), color='white')
        
        # Concatenate text fields
        full_text = self._concatenate_text(idx)
        
        # Process with CLIP processor
        inputs = self.processor(
            text=full_text,
            images=img,
            return_tensors="pt",
            padding="max_length",  # Consistent padding
            truncation=True,
            max_length=self.max_length
        )
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'price': self.prices[idx]
        }

In [3]:
class KNNFeatureAugmenter:
    def __init__(self, k=10, embedding_dim=512, use_gpu=False):
        """
        KNN-based feature augmentation for embeddings.
        
        Args:
            k: Number of nearest neighbors to consider
            embedding_dim: Dimension of the embeddings
            use_gpu: Whether to use GPU for FAISS operations (set to False for CPU-only)
        """
        self.k = k
        self.embedding_dim = embedding_dim
        self.use_gpu = use_gpu
        
        # Initialize FAISS indices for image and text embeddings
        self.img_index = None
        self.text_index = None
        
        # Store embeddings for meta-feature calculation (on CPU for FAISS)
        self.img_embeddings_cpu = None
        self.text_embeddings_cpu = None
        
        # Keep GPU versions for fast computation
        self.img_embeddings_gpu = None
        self.text_embeddings_gpu = None
        
    def _create_faiss_index(self, embeddings_cpu):
        """Create FAISS index for embeddings (CPU-only)."""
        # Ensure embeddings are on CPU and float32 for FAISS
        embeddings_np = embeddings_cpu.cpu().numpy().astype(np.float32)
        
        # Create CPU index (Inner product for cosine similarity with normalized vectors)
        index = faiss.IndexFlatIP(self.embedding_dim)
        
        # Add embeddings to index
        index.add(embeddings_np)
        
        return index
    
    def fit(self, img_embeddings, text_embeddings):
        """
        Build KNN indices from training embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [N, embedding_dim]
            text_embeddings: Text embeddings tensor [N, embedding_dim]
        """
        # Normalize embeddings for cosine similarity
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Store CPU versions for FAISS
        self.img_embeddings_cpu = img_embeddings_norm.cpu()
        self.text_embeddings_cpu = text_embeddings_norm.cpu()
        
        # Store GPU versions for fast computation (if available)
        if img_embeddings.is_cuda:
            self.img_embeddings_gpu = img_embeddings_norm
            self.text_embeddings_gpu = text_embeddings_norm
        else:
            self.img_embeddings_gpu = self.img_embeddings_cpu
            self.text_embeddings_gpu = self.text_embeddings_cpu
        
        # Create FAISS indices (CPU-only)
        self.img_index = self._create_faiss_index(self.img_embeddings_cpu)
        self.text_index = self._create_faiss_index(self.text_embeddings_cpu)
        
    def _calculate_knn_features(self, query_embeddings, index, stored_embeddings_gpu):
        """
        Calculate KNN-based meta-features for query embeddings.
        
        Returns:
            knn_features: Tensor of meta-features [batch_size, 4]
                - mean_similarity: Average similarity to k nearest neighbors
                - std_similarity: Standard deviation of similarities
                - min_similarity: Minimum similarity to k nearest neighbors
                - diversity_score: Measure of diversity among neighbors
        """
        # Move query to CPU for FAISS search
        query_np = query_embeddings.cpu().numpy().astype(np.float32)
        
        # Search for k+1 neighbors (including self)
        similarities, indices = index.search(query_np, self.k + 1)
        
        # Remove self-similarity (first result is usually the query itself)
        similarities = similarities[:, 1:]  # [batch_size, k]
        indices = indices[:, 1:]
        
        # Convert similarities back to GPU tensors
        similarities = torch.tensor(similarities, device=query_embeddings.device, dtype=query_embeddings.dtype)
        
        # Calculate meta-features on GPU
        mean_similarity = similarities.mean(dim=1)
        std_similarity = similarities.std(dim=1)
        min_similarity = similarities.min(dim=1)[0]
        
        # Calculate diversity score (average pairwise distance among neighbors)
        diversity_scores = []
        for i in range(len(indices)):
            neighbor_indices = indices[i]
            # Get neighbor embeddings (use GPU version if available)
            neighbor_embeddings = stored_embeddings_gpu[neighbor_indices]  # [k, embedding_dim]
            
            # Calculate pairwise cosine distances on GPU
            if len(neighbor_embeddings) > 1:
                similarity_matrix = torch.mm(neighbor_embeddings, neighbor_embeddings.t())
                # Get upper triangular part (excluding diagonal)
                triu_indices = torch.triu_indices(len(neighbor_embeddings), len(neighbor_embeddings), offset=1, device=neighbor_embeddings.device)
                pairwise_similarities = similarity_matrix[triu_indices[0], triu_indices[1]]
                diversity_score = 1.0 - pairwise_similarities.mean()  # Higher diversity = lower similarity
            else:
                diversity_score = torch.tensor(0.0, device=query_embeddings.device, dtype=query_embeddings.dtype)
            
            diversity_scores.append(diversity_score)
        
        diversity_scores = torch.stack(diversity_scores)
        
        # Stack all features
        knn_features = torch.stack([
            mean_similarity,
            std_similarity,
            min_similarity,
            diversity_scores
        ], dim=1)
        
        return knn_features
    
    def transform(self, img_embeddings, text_embeddings):
        """
        Calculate KNN meta-features for given embeddings.
        
        Args:
            img_embeddings: Image embeddings tensor [batch_size, embedding_dim]
            text_embeddings: Text embeddings tensor [batch_size, embedding_dim]
            
        Returns:
            img_knn_features: Image KNN meta-features [batch_size, 4]
            text_knn_features: Text KNN meta-features [batch_size, 4]
        """
        if self.img_index is None or self.text_index is None:
            raise ValueError("Must call fit() before transform()")
        
        # Normalize query embeddings
        img_embeddings_norm = torch.nn.functional.normalize(img_embeddings, p=2, dim=1)
        text_embeddings_norm = torch.nn.functional.normalize(text_embeddings, p=2, dim=1)
        
        # Calculate KNN features (CPU search, GPU computation)
        img_knn_features = self._calculate_knn_features(
            img_embeddings_norm, self.img_index, self.img_embeddings_gpu
        )
        text_knn_features = self._calculate_knn_features(
            text_embeddings_norm, self.text_index, self.text_embeddings_gpu
        )
        
        return img_knn_features, text_knn_features

In [4]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", k_neighbors=10):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get embedding dimension from CLIP
        self.embedding_dim = self.clip.config.projection_dim
        
        # Initialize KNN augmenter (CPU-only for FAISS)
        self.knn_augmenter = KNNFeatureAugmenter(
            k=k_neighbors, 
            embedding_dim=self.embedding_dim, 
            use_gpu=False  # Set to False for CPU-only FAISS
        )
        
        # Flag to track if KNN is fitted
        self.knn_fitted = False
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        return image_embeds, text_embeds
    
    def fit_knn(self, dataloader):
        """
        Fit KNN indices using all training data.
        
        Args:
            dataloader: DataLoader containing training samples
        """
        print("Fitting KNN indices...")
        all_img_embeds = []
        all_text_embeds = []
        
        self.eval()
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Extracting embeddings for KNN"):
                pixel_values = batch['pixel_values'].to(next(self.parameters()).device)
                input_ids = batch['input_ids'].to(next(self.parameters()).device)
                attention_mask = batch['attention_mask'].to(next(self.parameters()).device)
                
                img_embeds, text_embeds = self.extract_features(
                    pixel_values, input_ids, attention_mask
                )
                
                # Keep on GPU for now, will be handled in KNN augmenter
                all_img_embeds.append(img_embeds)
                all_text_embeds.append(text_embeds)
        
        # Concatenate all embeddings (keep on GPU)
        all_img_embeds = torch.cat(all_img_embeds, dim=0)
        all_text_embeds = torch.cat(all_text_embeds, dim=0)
        
        # Fit KNN augmenter (it will handle GPU-to-CPU movement internally)
        self.knn_augmenter.fit(all_img_embeds, all_text_embeds)
        self.knn_fitted = True
        print("KNN indices fitted successfully!")
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass that returns all four numerical features.
        
        Returns:
            dict with keys: 'text_embeds', 'img_embeds', 'text_knn', 'img_knn'
        """
        # Extract CLIP features
        with torch.no_grad():
            img_embeds, text_embeds = self.extract_features(
                pixel_values, input_ids, attention_mask
            )
        
        # Calculate KNN features if fitted
        if self.knn_fitted:
            img_knn_features, text_knn_features = self.knn_augmenter.transform(
                img_embeds, text_embeds
            )
        else:
            # Use zeros as placeholder if KNN not fitted
            batch_size = img_embeds.size(0)
            device = img_embeds.device
            img_knn_features = torch.zeros(batch_size, 4, device=device, dtype=img_embeds.dtype)
            text_knn_features = torch.zeros(batch_size, 4, device=device, dtype=text_embeds.dtype)
        
        return {
            'text_embeds': text_embeds,
            'img_embeds': img_embeds, 
            'text_knn': text_knn_features,
            'img_knn': img_knn_features
        }

In [5]:
def extract_and_save_features(model, dataloader, save_path, file_format='h5'):
    print(f"Extracting features and saving to {save_path}...")
    
    all_text_embeds = []
    all_img_embeds = []
    all_text_knn = []
    all_img_knn = []
    all_prices = []
    all_indices = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Extracting features")):
            pixel_values = batch['pixel_values'].to(next(model.parameters()).device)
            input_ids = batch['input_ids'].to(next(model.parameters()).device)
            attention_mask = batch['attention_mask'].to(next(model.parameters()).device)
            prices = batch['price']
            
            # Get all features
            features = model(pixel_values, input_ids, attention_mask)
            
            # Convert to CPU and collect
            all_text_embeds.append(features['text_embeds'].cpu().numpy())
            all_img_embeds.append(features['img_embeds'].cpu().numpy())
            all_text_knn.append(features['text_knn'].cpu().numpy())
            all_img_knn.append(features['img_knn'].cpu().numpy())
            all_prices.append(prices.numpy())
            
            # Track batch indices for reference
            batch_size = len(prices)
            start_idx = batch_idx * dataloader.batch_size
            batch_indices = list(range(start_idx, start_idx + batch_size))
            all_indices.extend(batch_indices)
    
    # Concatenate all features
    text_embeds = np.concatenate(all_text_embeds, axis=0)
    img_embeds = np.concatenate(all_img_embeds, axis=0)
    text_knn = np.concatenate(all_text_knn, axis=0)
    img_knn = np.concatenate(all_img_knn, axis=0)
    prices = np.concatenate(all_prices, axis=0)
    indices = np.array(all_indices)
    
    print(f"Feature shapes:")
    print(f"  text_embeds: {text_embeds.shape}")
    print(f"  img_embeds: {img_embeds.shape}")
    print(f"  text_knn: {text_knn.shape}")
    print(f"  img_knn: {img_knn.shape}")
    print(f"  prices: {prices.shape}")
    
    # Save based on format
    if file_format == 'h5':
        with h5py.File(save_path, 'w') as f:
            f.create_dataset('text_embeds', data=text_embeds, compression='gzip')
            f.create_dataset('img_embeds', data=img_embeds, compression='gzip')
            f.create_dataset('text_knn', data=text_knn, compression='gzip')
            f.create_dataset('img_knn', data=img_knn, compression='gzip')
            f.create_dataset('prices', data=prices, compression='gzip')
            f.create_dataset('indices', data=indices, compression='gzip')
            
            # Add metadata
            f.attrs['num_samples'] = len(prices)
            f.attrs['text_embed_dim'] = text_embeds.shape[1]
            f.attrs['img_embed_dim'] = img_embeds.shape[1]
            f.attrs['knn_features'] = 4
            
    elif file_format == 'npz':
        np.savez_compressed(
            save_path,
            text_embeds=text_embeds,
            img_embeds=img_embeds,
            text_knn=text_knn,
            img_knn=img_knn,
            prices=prices,
            indices=indices
        )
        
    elif file_format == 'pkl':
        features_dict = {
            'text_embeds': text_embeds,
            'img_embeds': img_embeds,
            'text_knn': text_knn,
            'img_knn': img_knn,
            'prices': prices,
            'indices': indices
        }
        with open(save_path, 'wb') as f:
            pickle.dump(features_dict, f)
    
    else:
        raise ValueError(f"Unsupported file format: {file_format}")
    
    print(f"Features saved successfully to {save_path}")
    return {
        'text_embeds': text_embeds,
        'img_embeds': img_embeds,
        'text_knn': text_knn,
        'img_knn': img_knn,
        'prices': prices,
        'indices': indices
    }

def load_features(file_path, file_format='h5'):
    if file_format == 'h5':
        features = {}
        with h5py.File(file_path, 'r') as f:
            features['text_embeds'] = f['text_embeds'][:]
            features['img_embeds'] = f['img_embeds'][:]
            features['text_knn'] = f['text_knn'][:]
            features['img_knn'] = f['img_knn'][:]
            features['prices'] = f['prices'][:]
            features['indices'] = f['indices'][:]
            
    elif file_format == 'npz':
        data = np.load(file_path)
        features = {
            'text_embeds': data['text_embeds'],
            'img_embeds': data['img_embeds'],
            'text_knn': data['text_knn'],
            'img_knn': data['img_knn'],
            'prices': data['prices'],
            'indices': data['indices']
        }
        
    elif file_format == 'pkl':
        with open(file_path, 'rb') as f:
            features = pickle.load(f)
    
    else:
        raise ValueError(f"Unsupported file format: {file_format}")
    
    return features

In [6]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

In [ ]:
for i in range(1, 4):
    TRAIN_IMG_DIR = f'../images/train_part{i}'
    VAL_IMG_DIR = f'../images/val_part{i}'
    train_df = pd.read_csv(f'../dataset/train_split/part{i}.csv')
    val_df = pd.read_csv(f'../dataset/val_split/part{i}.csv') 

    train_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=train_df['sample_id'].values,
        item_names=train_df['Item Name'].values,
        bullet_points=train_df['Bullet Points'].values,
        product_descriptions=train_df['Product Description'].values,
        values=train_df['Value'].values,
        units=train_df['Unit'].values,
        prices=train_df['log_price'].values,  
        processor=processor
    ),
    batch_size=5,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_dl = DataLoader(
    PriceDataset(
        img_path = VAL_IMG_DIR,
        image_paths=val_df['sample_id'].values,
        item_names=val_df['Item Name'].values,
        bullet_points=val_df['Bullet Points'].values,
        product_descriptions=val_df['Product Description'].values,
        values=val_df['Value'].values,
        units=val_df['Unit'].values,
        prices=val_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)    

model = CLIPPricePredictor(k_neighbors=10)
model.fit_knn(train_dl) 

train_features = extract_and_save_features(
    model, 
    train_dl, 
    f'train_features_{i}.h5',  # or .npz, .pkl
    file_format='h5'
)

val_features = extract_and_save_features(
    model, 
    val_dl, 
    f'val_features_{i}.h5',
    file_format='h5'
)

    # Clear GPU memory after each part
del model, train_dl, val_dl, train_features, val_features
torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!


Fitting KNN indices...


Extracting embeddings for KNN: 100%|██████████| 3997/3997 [07:57<00:00,  8.37it/s]



KNN indices fitted successfully!
Extracting features and saving to train_features_3.h5...


Extracting features: 100%|██████████| 3997/3997 [07:41<00:00,  8.67it/s]



Feature shapes:
  text_embeds: (19984, 512)
  img_embeds: (19984, 512)
  text_knn: (19984, 4)
  img_knn: (19984, 4)
  prices: (19984,)
Features saved successfully to train_features_3.h5
Extracting features and saving to val_features_3.h5...
Features saved successfully to train_features_3.h5
Extracting features and saving to val_features_3.h5...


Extracting features: 100%|██████████| 1249/1249 [02:03<00:00, 10.14it/s]



Feature shapes:
  text_embeds: (4996, 512)
  img_embeds: (4996, 512)
  text_knn: (4996, 4)
  img_knn: (4996, 4)
  prices: (4996,)
Features saved successfully to val_features_3.h5


In [ ]:
# 4. Load features later
loaded_features = load_features('train_features.h5', file_format='h5')
print("Loaded features shapes:")
for key, value in loaded_features.items():
    print(f"  {key}: {value.shape}")

Fitting KNN indices...


Extracting embeddings for KNN: 100%|██████████| 4996/4996 [07:23<00:00, 11.28it/s]



KNN indices fitted successfully!
Extracting features and saving to train_features_3.h5...


Extracting features: 100%|██████████| 4996/4996 [07:50<00:00, 10.61it/s]



Feature shapes:
  text_embeds: (19984, 512)
  img_embeds: (19984, 512)
  text_knn: (19984, 4)
  img_knn: (19984, 4)
  prices: (19984,)
Features saved successfully to train_features_3.h5
Extracting features and saving to val_features_3.h5...
Features saved successfully to train_features_3.h5
Extracting features and saving to val_features_3.h5...


Extracting features: 100%|██████████| 1249/1249 [02:01<00:00, 10.31it/s]



Feature shapes:
  text_embeds: (4996, 512)
  img_embeds: (4996, 512)
  text_knn: (4996, 4)
  img_knn: (4996, 4)
  prices: (4996,)
Features saved successfully to val_features_3.h5
Loaded features shapes:
  text_embeds: (19589, 512)
  img_embeds: (19589, 512)
  text_knn: (19589, 4)
  img_knn: (19589, 4)
  prices: (19589,)
  indices: (19589,)
Loaded features shapes:
  text_embeds: (19589, 512)
  img_embeds: (19589, 512)
  text_knn: (19589, 4)
  img_knn: (19589, 4)
  prices: (19589,)
  indices: (19589,)


## XGBoost Approach

In [1]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import h5py
import gc
import time

In [ ]:
class XGBFeatureTrainer:
    def __init__(self, use_gpu=True, gpu_id=0):
        """
        XGBoost trainer for multimodal features with GPU acceleration.
        
        Args:
            use_gpu: Whether to use GPU for training
            gpu_id: GPU device ID to use
        """
        self.use_gpu = use_gpu
        self.gpu_id = gpu_id
        self.model = None
        self.feature_names = ['text_embeds', 'img_embeds', 'text_knn', 'img_knn']
        
    def load_and_prepare_data(self, h5_file_path):
        """
        Load features from H5 file and concatenate them.
        
        Args:
            h5_file_path: Path to the H5 file containing features
            
        Returns:
            X: Combined feature matrix [N, total_features]
            y: Target prices [N]
            feature_info: Dictionary with feature dimensions
        """
        print(f"Loading data from {h5_file_path}...")
        
        with h5py.File(h5_file_path, 'r') as f:
            # Load all features
            text_embeds = f['text_embeds'][:]
            img_embeds = f['img_embeds'][:]
            text_knn = f['text_knn'][:]
            img_knn = f['img_knn'][:]
            prices = f['prices'][:]
            
            print(f"Loaded feature shapes:")
            print(f"  text_embeds: {text_embeds.shape}")
            print(f"  img_embeds: {img_embeds.shape}")
            print(f"  text_knn: {text_knn.shape}")
            print(f"  img_knn: {img_knn.shape}")
            print(f"  prices: {prices.shape}")
        
        # Concatenate all features
        X = np.concatenate([text_embeds, img_embeds, text_knn, img_knn], axis=1)
        y = prices
        
        # Feature information for analysis
        feature_info = {
            'text_embeds_dim': text_embeds.shape[1],
            'img_embeds_dim': img_embeds.shape[1],
            'text_knn_dim': text_knn.shape[1],
            'img_knn_dim': img_knn.shape[1],
            'total_features': X.shape[1],
            'num_samples': X.shape[0]
        }
        
        print(f"Combined feature matrix shape: {X.shape}")
        print(f"Target shape: {y.shape}")
        
        # Clean up memory
        del text_embeds, img_embeds, text_knn, img_knn
        gc.collect()
        
        return X, y, feature_info
    
    def create_xgb_params(self, feature_info, learning_rate=0.1, max_depth=6, 
                         n_estimators=1000, subsample=0.8, colsample_bytree=0.8):
        params = {
            # GPU Configuration
            'tree_method': 'gpu_hist' if self.use_gpu else 'hist',
            'gpu_id': self.gpu_id if self.use_gpu else None,
            'predictor': 'gpu_predictor' if self.use_gpu else 'cpu_predictor',
            
            # Model Parameters
            'objective': 'reg:squarederror',
            'eval_metric': ['rmse', 'mae'],
            'learning_rate': learning_rate,
            'max_depth': max_depth,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'min_child_weight': 3,
            'gamma': 0.1,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            
            # Performance
            'n_jobs': -1,
            'random_state': 42,
            'verbosity': 1,
            
            # Memory optimization for large feature sets
            'max_bin': 256,
            'grow_policy': 'depthwise'
        }
        
        # Remove None values
        params = {k: v for k, v in params.items() if v is not None}
        
        print("XGBoost Parameters:")
        for key, value in params.items():
            print(f"  {key}: {value}")
        
        return params, n_estimators
    
    def train(self, train_h5_path, val_h5_path, 
              learning_rate=0.1, max_depth=6, n_estimators=1000,
              early_stopping_rounds=50, verbose_eval=100):
        print("=" * 60)
        print("TRAINING XGBOOST MODEL WITH GPU ACCELERATION")
        print("=" * 60)
        
        # Load training data
        X_train, y_train, train_info = self.load_and_prepare_data(train_h5_path)
        
        # Load validation data
        X_val, y_val, val_info = self.load_and_prepare_data(val_h5_path)
        
        # Verify feature dimensions match
        assert X_train.shape[1] == X_val.shape[1], "Feature dimensions don't match!"
        
        # Create XGBoost parameters
        params, n_est = self.create_xgb_params(
            train_info, learning_rate, max_depth, n_estimators
        )
        
        # Create DMatrix for GPU acceleration
        print("\nCreating DMatrix for GPU training...")
        dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=[f'feat_{i}' for i in range(X_train.shape[1])])
        dval = xgb.DMatrix(X_val, label=y_val, feature_names=[f'feat_{i}' for i in range(X_val.shape[1])])
        
        # Set up evaluation
        evallist = [(dtrain, 'train'), (dval, 'val')]
        
        # Train model
        print(f"\nStarting training with {n_est} estimators...")
        start_time = time.time()
        
        # Store training history
        evals_result = {}
        
        self.model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=n_est,
            evals=evallist,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=verbose_eval,
            evals_result=evals_result
        )
        
        training_time = time.time() - start_time
        print(f"\nTraining completed in {training_time:.2f} seconds")
        
        # Evaluate model
        print("\n" + "=" * 40)
        print("MODEL EVALUATION")
        print("=" * 40)
        
        # Predictions
        train_pred = self.model.predict(dtrain)
        val_pred = self.model.predict(dval)
        
        # Calculate metrics
        train_metrics = self._calculate_metrics(y_train, train_pred, "Training")
        val_metrics = self._calculate_metrics(y_val, val_pred, "Validation")
        
        # Feature importance
        importance = self.model.get_score(importance_type='weight')
        
        # Clean up memory
        del X_train, X_val, y_train, y_val, dtrain, dval
        gc.collect()
        
        return {
            'model': self.model,
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'feature_importance': importance,
            'training_time': training_time,
            'evals_result': evals_result,
            'feature_info': train_info
        }
    
    def _calculate_metrics(self, y_true, y_pred, dataset_name):
        """Calculate and print regression metrics."""
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        print(f"\n{dataset_name} Metrics:")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  MAE:  {mae:.4f}")
        print(f"  R²:   {r2:.4f}")
        
        return {
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'mse': mse
        }
    
    def save_model(self, model_path):
        """Save the trained model."""
        if self.model is None:
            raise ValueError("No model to save. Train a model first.")
        
        self.model.save_model(model_path)
        print(f"Model saved to {model_path}")
    
    def load_model(self, model_path):
        """Load a pre-trained model."""
        self.model = xgb.Booster()
        self.model.load_model(model_path)
        print(f"Model loaded from {model_path}")
    
    def predict(self, h5_file_path):
        """Make predictions on new data."""
        if self.model is None:
            raise ValueError("No model loaded. Train or load a model first.")
        
        X, _, _ = self.load_and_prepare_data(h5_file_path)
        dtest = xgb.DMatrix(X, feature_names=[f'feat_{i}' for i in range(X.shape[1])])
        predictions = self.model.predict(dtest)
        
        del X, dtest
        gc.collect()
        
        return predictions
    
    def analyze_feature_importance(self, top_n=20):
        """Analyze and display feature importance."""
        if self.model is None:
            raise ValueError("No model to analyze. Train a model first.")
        
        # Get feature importance
        importance = self.model.get_score(importance_type='weight')
        
        # Sort by importance
        sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nTop {top_n} Most Important Features:")
        print("-" * 40)
        for i, (feature, score) in enumerate(sorted_importance[:top_n]):
            feature_idx = int(feature.split('_')[1])
            feature_type = self._get_feature_type(feature_idx)
            print(f"{i+1:2d}. {feature} ({feature_type}): {score}")
        
        return sorted_importance
    
    def _get_feature_type(self, feature_idx):
        """Determine which feature type an index belongs to."""
        # This is a simplified version - you might want to store the actual dimensions
        if feature_idx < 512:  # Assuming CLIP embedding dimension
            return "text_embed"
        elif feature_idx < 1024:
            return "img_embed"
        elif feature_idx < 1028:
            return "text_knn"
        else:
            return "img_knn"

In [ ]:
def train_xgboost_model():
    trainer = XGBFeatureTrainer(use_gpu=True, gpu_id=0)
    
    # Train the model
    results = trainer.train(
        train_h5_path='train_features.h5',
        val_h5_path='val_features.h5',
        learning_rate=0.1,
        max_depth=8,
        n_estimators=2000,
        early_stopping_rounds=100,
        verbose_eval=50
    )
    
    # Save the model
    trainer.save_model('xgb_price_model.json')
    
    # Analyze feature importance
    importance = trainer.analyze_feature_importance(top_n=30)
    
    return trainer, results

# Run the training
trainer, results = train_xgboost_model()

TRAINING XGBOOST MODEL WITH GPU ACCELERATION
Loading data from train_features.h5...
Loaded feature shapes:
  text_embeds: (19589, 512)
  img_embeds: (19589, 512)
  text_knn: (19589, 4)
  img_knn: (19589, 4)
  prices: (19589,)
Combined feature matrix shape: (19589, 1032)
Target shape: (19589,)
Loading data from val_features.h5...
Loaded feature shapes:
  text_embeds: (4980, 512)
  img_embeds: (4980, 512)
  text_knn: (4980, 4)
  img_knn: (4980, 4)
  prices: (4980,)
Combined feature matrix shape: (4980, 1032)
Target shape: (4980,)
Loading data from val_features.h5...
Loaded feature shapes:
  text_embeds: (4980, 512)
  img_embeds: (4980, 512)
  text_knn: (4980, 4)
  img_knn: (4980, 4)
  prices: (4980,)
Combined feature matrix shape: (4980, 1032)
Target shape: (4980,)
XGBoost Parameters:
  tree_method: gpu_hist
  gpu_id: 0
  predictor: gpu_predictor
  objective: reg:squarederror
  eval_metric: ['rmse', 'mae']
  learning_rate: 0.1
  max_depth: 8
  subsample: 0.8
  colsample_bytree: 0.8
  min

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [11:23:15] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  self.starting_round = model.num_boosted_rounds()
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [11:23:15] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [11:23:15] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:0.90960	train-mae:0.73538	val-rmse:0.93868	val-mae:0.75797
[50]	train-rmse:0.42985	train-mae:0.33843	val-rmse:0.79572	val-mae:0.61945
[50]	train-rmse:0.42985	train-mae:0.33843	val-rmse:0.79572	val-mae:0.61945
[100]	train-rmse:0.27549	train-mae:0.21511	val-rmse:0.78692	val-mae:0.61035
[100]	train-rmse:0.27549	train-mae:0.21511	val-rmse:0.78692	val-mae:0.61035
[150]	train-rmse:0.18959	train-mae:0.14699	val-rmse:0.78289	val-mae:0.60637
[150]	train-rmse:0.18959	train-mae:0.14699	val-rmse:0.78289	val-mae:0.60637
[200]	train-rmse:0.13175	train-mae:0.10148	val-rmse:0.78090	val-mae:0.60488
[200]	train-rmse:0.13175	train-mae:0.10148	val-rmse:0.78090	val-mae:0.60488
[250]	train-rmse:0.10155	train-mae:0.07814	val-rmse:0.78027	val-mae:0.60399
[250]	train-rmse:0.10155	train-mae:0.07814	val-rmse:0.78027	val-mae:0.60399
[300]	train-rmse:0.09576	train-mae:0.07371	val-rmse:0.78011	val-mae:0.60371
[300]	train-rmse:0.09576	train-mae:0.07371	val-rmse:0.78011	val-mae:0.60371
[350]	train-rmse

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/core.py:729: UserWarning: [11:23:35] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)


Model saved to xgb_price_model.json

Top 30 Most Important Features:
----------------------------------------
 1. feat_1 (text_embed): 67.0
 2. feat_2 (text_embed): 64.0
 3. feat_442 (text_embed): 63.0
 4. feat_0 (text_embed): 60.0
 5. feat_355 (text_embed): 58.0
 6. feat_340 (text_embed): 57.0
 7. feat_723 (img_embed): 57.0
 8. feat_903 (img_embed): 57.0
 9. feat_234 (text_embed): 56.0
10. feat_649 (img_embed): 56.0
11. feat_86 (text_embed): 55.0
12. feat_156 (text_embed): 55.0
13. feat_379 (text_embed): 55.0
14. feat_3 (text_embed): 54.0
15. feat_7 (text_embed): 54.0
16. feat_319 (text_embed): 54.0
17. feat_321 (text_embed): 54.0
18. feat_653 (img_embed): 54.0
19. feat_9 (text_embed): 53.0
20. feat_22 (text_embed): 53.0
21. feat_96 (text_embed): 53.0
22. feat_336 (text_embed): 53.0
23. feat_124 (text_embed): 52.0
24. feat_184 (text_embed): 52.0
25. feat_634 (img_embed): 52.0
26. feat_640 (img_embed): 52.0
27. feat_1029 (img_knn): 52.0
28. feat_10 (text_embed): 51.0
29. feat_17 (text_

In [9]:
def retrain_xgboost_model(existing_model_path, new_h5_files, val_h5_path, 
                         learning_rate=0.05, max_depth=8, n_estimators=1000,
                         early_stopping_rounds=100, verbose_eval=50):
    print("=" * 60)
    print("RETRAINING XGBOOST MODEL ON ADDITIONAL DATA")
    print("=" * 60)
    
    # Initialize trainer
    trainer = XGBFeatureTrainer(use_gpu=True, gpu_id=0)
    
    # Load existing model
    print(f"Loading existing model from {existing_model_path}...")
    trainer.load_model(existing_model_path)
    
    # Load and combine all training data
    print("Loading and combining training data...")
    all_X_train = []
    all_y_train = []
    
    for i, h5_file in enumerate(new_h5_files):
        print(f"Loading {h5_file} ({i+1}/{len(new_h5_files)})...")
        X, y, feature_info = trainer.load_and_prepare_data(h5_file)
        all_X_train.append(X)
        all_y_train.append(y)
        
        # Clean up immediately to save memory
        del X, y
        gc.collect()
    
    # Concatenate all training data
    print("Concatenating training data...")
    X_train_combined = np.concatenate(all_X_train, axis=0)
    y_train_combined = np.concatenate(all_y_train, axis=0)
    
    # Clean up
    del all_X_train, all_y_train
    gc.collect()
    
    print(f"Combined training data shape: {X_train_combined.shape}")
    print(f"Combined training targets shape: {y_train_combined.shape}")
    
    # Load validation data
    print("Loading and combining Validation data...")
    all_X_val = []
    all_y_val = []
    X_val, y_val, val_info = trainer.load_and_prepare_data(val_h5_path)
    for i, h5_file in enumerate(val_h5_path):
        print(f"Loading {h5_file} ({i+1}/{len(val_h5_path)})...")
        X, y, feature_info = trainer.load_and_prepare_data(h5_file)
        all_X_val.append(X)
        all_y_val.append(y)
        
        del X, y
        gc.collect()
    
    # Concatenate all training data
    print("Concatenating training data...")
    X_val_combined = np.concatenate(all_X_val, axis=0)
    y_val_combined = np.concatenate(all_y_val, axis=0)
    
    # Clean up
    del all_X_train, all_y_train
    gc.collect()
    
    # Verify feature dimensions match
    assert X_train_combined.shape[1] == X_val_combined.shape[1], "Feature dimensions don't match!"
    
    # Create XGBoost parameters for retraining
    params = {
        'tree_method': 'gpu_hist',
        'gpu_id': 0,
        'predictor': 'gpu_predictor',
        'objective': 'reg:squarederror',
        'eval_metric': ['rmse', 'mae'],
        'learning_rate': learning_rate,  # Usually lower for retraining
        'max_depth': max_depth,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_jobs': -1,
        'random_state': 42,
        'verbosity': 1,
        'max_bin': 256,
        'grow_policy': 'depthwise'
    }
    
    print("\nRetraining Parameters:")
    for key, value in params.items():
        print(f"  {key}: {value}")
    
    # Create DMatrix for GPU acceleration
    print("\nCreating DMatrix for GPU retraining...")
    dtrain = xgb.DMatrix(X_train_combined, label=y_train_combined, 
                        feature_names=[f'feat_{i}' for i in range(X_train_combined.shape[1])])
    dval = xgb.DMatrix(X_val_combined, label=y_val_combined, 
                      feature_names=[f'feat_{i}' for i in range(X_val_combined.shape[1])])
    
    # Set up evaluation
    evallist = [(dtrain, 'train'), (dval, 'val')]
    
    # Continue training from existing model
    print(f"\nContinuing training for {n_estimators} additional rounds...")
    start_time = time.time()
    
    # Store training history
    evals_result = {}
    
    # Use xgb.train with existing model as starting point
    trainer.model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_estimators,
        evals=evallist,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=verbose_eval,
        evals_result=evals_result,
        xgb_model=trainer.model  # Continue from existing model
    )
    
    training_time = time.time() - start_time
    print(f"\nRetraining completed in {training_time:.2f} seconds")
    
    # Evaluate retrained model
    print("\n" + "=" * 40)
    print("RETRAINED MODEL EVALUATION")
    print("=" * 40)
    
    # Predictions
    train_pred = trainer.model.predict(dtrain)
    val_pred = trainer.model.predict(dval)
    
    # Calculate metrics
    train_metrics = trainer._calculate_metrics(y_train_combined, train_pred, "Training")
    val_metrics = trainer._calculate_metrics(y_val_combined, val_pred, "Validation")
    
    # Feature importance
    importance = trainer.model.get_score(importance_type='weight')
    
    # Clean up memory
    del X_train_combined, X_val, y_train_combined, y_val, dtrain, dval
    gc.collect()
    
    return {
        'trainer': trainer,
        'model': trainer.model,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'feature_importance': importance,
        'training_time': training_time,
        'evals_result': evals_result,
        'feature_info': feature_info
    }

In [10]:
def incremental_train_xgboost(existing_model_path, new_h5_files, val_h5_files,
                             save_path='retrained_xgb_model.json'):
    print("INCREMENTAL XGBOOST RETRAINING")
    
    trainer = XGBFeatureTrainer(use_gpu=True, gpu_id=0)
    trainer.load_model(existing_model_path)
    
    print("Loading and combining validation data...")
    all_X_val = []
    all_y_val = []
    
    for i, val_file in enumerate(val_h5_files):
        print(f"Loading validation file {val_file} ({i+1}/{len(val_h5_files)})...")
        X_val, y_val, _ = trainer.load_and_prepare_data(val_file)
        all_X_val.append(X_val)
        all_y_val.append(y_val)
        
        # Clean up immediately to save memory
        del X_val, y_val
        gc.collect()
    
    # Concatenate all validation data
    print("Concatenating validation data...")
    X_val_combined = np.concatenate(all_X_val, axis=0)
    y_val_combined = np.concatenate(all_y_val, axis=0)
    
    # Clean up
    del all_X_val, all_y_val
    gc.collect()
    
    print(f"Combined validation data shape: {X_val_combined.shape}")
    print(f"Combined validation targets shape: {y_val_combined.shape}")
    
    # Create validation DMatrix
    dval = xgb.DMatrix(X_val_combined, label=y_val_combined, 
                      feature_names=[f'feat_{i}' for i in range(X_val_combined.shape[1])])
    
    # Incremental training parameters
    params = {
        'tree_method': 'gpu_hist',
        'gpu_id': 0,
        'predictor': 'gpu_predictor',
        'objective': 'reg:squarederror',
        'eval_metric': ['rmse', 'mae'],
        'learning_rate': 0.05,  # Lower learning rate for incremental training
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'n_jobs': -1,
        'random_state': 42,
        'verbosity': 1
    }
    
    # Train on each file incrementally
    for i, h5_file in enumerate(new_h5_files):
        print(f"\nIncremental training on {h5_file} ({i+1}/{len(new_h5_files)})...")
        
        # Load current training data
        X_train, y_train, _ = trainer.load_and_prepare_data(h5_file)
        dtrain = xgb.DMatrix(X_train, label=y_train,
                           feature_names=[f'feat_{i}' for i in range(X_train.shape[1])])
        
        # Set up evaluation
        evallist = [(dtrain, 'train'), (dval, 'val')]
        
        # Continue training
        evals_result = {}
        trainer.model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=500,  # Fewer rounds per file
            evals=evallist,
            early_stopping_rounds=50,
            verbose_eval=100,
            evals_result=evals_result,
            xgb_model=trainer.model
        )
        
        # Clean up
        del X_train, y_train, dtrain
        gc.collect()
        
        # Evaluate current performance
        val_pred = trainer.model.predict(dval)
        val_rmse = np.sqrt(mean_squared_error(y_val_combined, val_pred))
        print(f"Validation RMSE after file {i+1}: {val_rmse:.4f}")
    
    # Final evaluation
    print("\n" + "=" * 40)
    print("FINAL INCREMENTAL MODEL EVALUATION")
    print("=" * 40)
    
    val_pred = trainer.model.predict(dval)
    final_metrics = trainer._calculate_metrics(y_val_combined, val_pred, "Final Validation")
    
    # Save retrained model
    trainer.save_model(save_path)
    
    # Clean up
    del X_val_combined, y_val_combined, dval
    gc.collect()
    
    return trainer, final_metrics

In [11]:
def main_retrain():
    """Main function to retrain XGBoost model on additional data."""
    
    existing_model_path = 'xgb_price_model.json'
    new_training_files = [
        'train_features_2.h5',
        'train_features_3.h5'
    ]
    val_files = [
        'val_features_2.h5',
        'val_features_3.h5'
    ]
    
    print("Starting XGBoost retraining process...")
    print(f"Existing model: {existing_model_path}")
    print(f"New training files: {new_training_files}")
    print(f"Validation files: {val_files}")
    
    # Method 1: Full retraining (combines all new data)
    print("\n" + "="*60)
    print("METHOD 1: FULL RETRAINING")
    print("="*60)
    
    try:
        results_full = retrain_xgboost_model(
            existing_model_path=existing_model_path,
            new_h5_files=new_training_files,
            val_h5_path=val_files,  # Pass list of validation files
            learning_rate=0.05,  # Lower learning rate for stability
            max_depth=8,
            n_estimators=1000,
            early_stopping_rounds=100,
            verbose_eval=50
        )
        
        # Save the retrained model
        results_full['trainer'].save_model('retrained_xgb_full.json')
        
        # Analyze feature importance
        print("\nAnalyzing feature importance after retraining...")
        importance = results_full['trainer'].analyze_feature_importance(top_n=30)
        
        print("\nFull retraining completed successfully!")
        
        return results_full['trainer'], results_full
        
    except Exception as e:
        print(f"Full retraining failed: {e}")
        print("Trying incremental approach...")
        
        # Method 2: Incremental retraining (if memory is limited)
        print("\n" + "="*60)
        print("METHOD 2: INCREMENTAL RETRAINING")
        print("="*60)
        
        trainer_inc, metrics_inc = incremental_train_xgboost(
            existing_model_path=existing_model_path,
            new_h5_files=new_training_files,
            val_h5_files=val_files,  # Now properly handles multiple validation files
            save_path='retrained_xgb_incremental.json'
        )
        
        print("\nIncremental retraining completed successfully!")
        
        return trainer_inc, metrics_inc

# Execute the retraining
retrained_model, final_results = main_retrain()

Starting XGBoost retraining process...
Existing model: xgb_price_model.json
New training files: ['train_features_2.h5', 'train_features_3.h5']
Validation files: ['val_features_2.h5', 'val_features_3.h5']

METHOD 1: FULL RETRAINING
RETRAINING XGBOOST MODEL ON ADDITIONAL DATA
Loading existing model from xgb_price_model.json...
Model loaded from xgb_price_model.json
Loading and combining training data...
Loading train_features_2.h5 (1/2)...
Loading data from train_features_2.h5...
Loaded feature shapes:
  text_embeds: (19754, 512)
  img_embeds: (19754, 512)
  text_knn: (19754, 4)
  img_knn: (19754, 4)
  prices: (19754,)
Combined feature matrix shape: (19754, 1032)
Target shape: (19754,)
Loaded feature shapes:
  text_embeds: (19754, 512)
  img_embeds: (19754, 512)
  text_knn: (19754, 4)
  img_knn: (19754, 4)
  prices: (19754,)
Combined feature matrix shape: (19754, 1032)
Target shape: (19754,)
Loading train_features_3.h5 (2/2)...
Loading data from train_features_3.h5...
Loaded feature shap

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [12:14:58] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [12:14:58] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:0.75020	train-mae:0.58472	val-rmse:0.76610	val-mae:0.59791
[100]	train-rmse:0.47023	train-mae:0.37001	val-rmse:0.75438	val-mae:0.58637
[100]	train-rmse:0.47023	train-mae:0.37001	val-rmse:0.75438	val-mae:0.58637
[200]	train-rmse:0.31850	train-mae:0.25045	val-rmse:0.75140	val-mae:0.58347
[200]	train-rmse:0.31850	train-mae:0.25045	val-rmse:0.75140	val-mae:0.58347
[300]	train-rmse:0.21829	train-mae:0.17068	val-rmse:0.74994	val-mae:0.58158
[300]	train-rmse:0.21829	train-mae:0.17068	val-rmse:0.74994	val-mae:0.58158
[400]	train-rmse:0.15311	train-mae:0.11887	val-rmse:0.74930	val-mae:0.58052
[400]	train-rmse:0.15311	train-mae:0.11887	val-rmse:0.74930	val-mae:0.58052
[499]	train-rmse:0.11165	train-mae:0.08626	val-rmse:0.74875	val-mae:0.57981
[499]	train-rmse:0.11165	train-mae:0.08626	val-rmse:0.74875	val-mae:0.57981


/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/core.py:729: UserWarning: [12:15:32] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)


Validation RMSE after file 1: 0.7488

Incremental training on train_features_3.h5 (2/2)...
Loading data from train_features_3.h5...
Loaded feature shapes:
  text_embeds: (19984, 512)
  img_embeds: (19984, 512)
  text_knn: (19984, 4)
  img_knn: (19984, 4)
  prices: (19984,)
Combined feature matrix shape: (19984, 1032)
Target shape: (19984,)
Combined feature matrix shape: (19984, 1032)
Target shape: (19984,)


/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [12:15:32] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [12:15:32] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:0.74794	train-mae:0.57732	val-rmse:0.74880	val-mae:0.57985
[100]	train-rmse:0.49330	train-mae:0.38659	val-rmse:0.74603	val-mae:0.57595
[100]	train-rmse:0.49330	train-mae:0.38659	val-rmse:0.74603	val-mae:0.57595
[200]	train-rmse:0.34048	train-mae:0.26689	val-rmse:0.74522	val-mae:0.57474
[200]	train-rmse:0.34048	train-mae:0.26689	val-rmse:0.74522	val-mae:0.57474
[233]	train-rmse:0.30686	train-mae:0.24038	val-rmse:0.74561	val-mae:0.57497
[233]	train-rmse:0.30686	train-mae:0.24038	val-rmse:0.74561	val-mae:0.57497


/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/xgboost/core.py:729: UserWarning: [12:15:49] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)


Validation RMSE after file 2: 0.7456

FINAL INCREMENTAL MODEL EVALUATION

Final Validation Metrics:
  RMSE: 0.7456
  MAE:  0.5750
  R²:   0.3825
Model saved to retrained_xgb_incremental.json

Incremental retraining completed successfully!

Incremental retraining completed successfully!


## Light BGM Approach

In [1]:
import gc
gc.collect()
torch.cuda.empty_cache()


NameError: name 'torch' is not defined

In [2]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import h5py
import gc
import time

In [6]:
class LGBMFeatureTrainer:
    def __init__(self, use_gpu=True, gpu_id=0):
        """
        LightGBM trainer for multimodal features with GPU acceleration.
        
        Args:
            use_gpu: Whether to use GPU for training
            gpu_id: GPU device ID to use
        """
        self.use_gpu = use_gpu
        self.gpu_id = gpu_id
        self.model = None
        self.feature_names = ['text_embeds', 'img_embeds', 'text_knn', 'img_knn']
        
    def load_and_prepare_data(self, h5_file_path):
        """
        Load features from H5 file and concatenate them.
        
        Args:
            h5_file_path: Path to the H5 file containing features
            
        Returns:
            X: Combined feature matrix [N, total_features]
            y: Target prices [N]
            feature_info: Dictionary with feature dimensions
        """
        print(f"Loading data from {h5_file_path}...")
        
        with h5py.File(h5_file_path, 'r') as f:
            # Load all features
            text_embeds = f['text_embeds'][:]
            img_embeds = f['img_embeds'][:]
            text_knn = f['text_knn'][:]
            img_knn = f['img_knn'][:]
            prices = f['prices'][:]
            
            print(f"Loaded feature shapes:")
            print(f"  text_embeds: {text_embeds.shape}")
            print(f"  img_embeds: {img_embeds.shape}")
            print(f"  text_knn: {text_knn.shape}")
            print(f"  img_knn: {img_knn.shape}")
            print(f"  prices: {prices.shape}")
        
        # Concatenate all features
        X = np.concatenate([text_embeds, img_embeds, text_knn, img_knn], axis=1)
        y = prices
        
        # Feature information for analysis
        feature_info = {
            'text_embeds_dim': text_embeds.shape[1],
            'img_embeds_dim': img_embeds.shape[1],
            'text_knn_dim': text_knn.shape[1],
            'img_knn_dim': img_knn.shape[1],
            'total_features': X.shape[1],
            'num_samples': X.shape[0]
        }
        
        print(f"Combined feature matrix shape: {X.shape}")
        print(f"Target shape: {y.shape}")
        
        # Clean up memory
        del text_embeds, img_embeds, text_knn, img_knn
        gc.collect()
        
        return X, y, feature_info
    
    def create_lgb_params(self, feature_info, learning_rate=0.1, max_depth=10, 
                         n_estimators=1000, subsample=0.8, colsample_bytree=0.8):
        """Create LightGBM parameters with strong regularization to prevent overfitting."""
        params = {
            # Core parameters
            'objective': 'regression',
            'metric': ['rmse', 'mae'],
            'boosting_type': 'gbdt',
            'learning_rate': learning_rate,
            'num_leaves': min(31, 2 ** max_depth - 1),  # Control complexity
            'max_depth': max_depth,
            
            # Regularization parameters (key for preventing overfitting)
            'min_data_in_leaf': 30,  # Minimum samples per leaf
            'min_sum_hessian_in_leaf': 5e-3,  # Minimum hessian sum per leaf
            'feature_fraction': colsample_bytree,  # Feature sampling
            'bagging_fraction': subsample,  # Row sampling
            'bagging_freq': 10,  # Frequency of bagging
            'lambda_l1': 0.1,  # L1 regularization
            'lambda_l2': 0.1,  # L2 regularization
            'min_gain_to_split': 0.02,  # Minimum gain to split
            
            # Performance parameters
            'verbosity': 1,
            'seed': 42,
            'deterministic': True,
            'force_row_wise': True,  # For better performance
            
            # GPU parameters
            'device': 'gpu' if self.use_gpu else 'cpu',
            'gpu_platform_id': 0 if self.use_gpu else -1,
            'gpu_device_id': self.gpu_id if self.use_gpu else -1,
            'num_threads': -1,
            
            # Early stopping prevention
            'early_stopping_round': 100,
            'first_metric_only': True,
        }
        
        # Remove GPU params if not using GPU
        if not self.use_gpu:
            params.pop('gpu_platform_id', None)
            params.pop('gpu_device_id', None)
        
        print("LightGBM Parameters:")
        for key, value in params.items():
            print(f"  {key}: {value}")
        
        return params, n_estimators
    
    def train(self, train_h5_path, val_h5_path, 
              learning_rate=0.05, max_depth=6, n_estimators=1000,
              early_stopping_rounds=100, verbose_eval=100):
        """Train initial LightGBM model."""
        print("=" * 60)
        print("TRAINING LIGHTGBM MODEL WITH REGULARIZATION")
        print("=" * 60)
        
        # Load training data
        X_train, y_train, train_info = self.load_and_prepare_data(train_h5_path)
        
        # Load validation data
        X_val, y_val, val_info = self.load_and_prepare_data(val_h5_path)
        
        # Verify feature dimensions match
        assert X_train.shape[1] == X_val.shape[1], "Feature dimensions don't match!"
        
        # Create LightGBM parameters
        params, n_est = self.create_lgb_params(
            train_info, learning_rate, max_depth, n_estimators
        )
        
        # Create LightGBM datasets - KEY FIX: free_raw_data=False
        print("\nCreating LightGBM datasets...")
        train_data = lgb.Dataset(
            X_train, 
            label=y_train, 
            feature_name=[f'feat_{i}' for i in range(X_train.shape[1])],
            free_raw_data=False  # CRITICAL: Keep raw data for predictions
        )
        val_data = lgb.Dataset(
            X_val, 
            label=y_val, 
            reference=train_data,
            feature_name=[f'feat_{i}' for i in range(X_val.shape[1])],
            free_raw_data=False  # CRITICAL: Keep raw data for predictions
        )
        
        # Train model
        print(f"\nStarting training with {n_est} estimators...")
        start_time = time.time()
        
        # Store training history
        evals_result = {}
        
        self.model = lgb.train(
            params=params,
            train_set=train_data,
            num_boost_round=n_est,
            valid_sets=[train_data, val_data],
            valid_names=['train', 'val'],
            callbacks=[
                lgb.early_stopping(early_stopping_rounds),
                lgb.log_evaluation(verbose_eval),
                lgb.record_evaluation(evals_result)
            ]
        )
        
        training_time = time.time() - start_time
        print(f"\nTraining completed in {training_time:.2f} seconds")
        print(f"Best iteration: {self.model.best_iteration}")
        
        # Evaluate model
        print("\n" + "=" * 40)
        print("MODEL EVALUATION")
        print("=" * 40)
        
        # Predictions
        train_pred = self.model.predict(X_train, num_iteration=self.model.best_iteration)
        val_pred = self.model.predict(X_val, num_iteration=self.model.best_iteration)
        
        # Calculate metrics
        train_metrics = self._calculate_metrics(y_train, train_pred, "Training")
        val_metrics = self._calculate_metrics(y_val, val_pred, "Validation")
        
        # Feature importance
        importance = self.model.feature_importance(importance_type='gain')
        
        # Clean up memory
        del X_train, X_val, y_train, y_val, train_data, val_data
        gc.collect()
        
        return {
            'model': self.model,
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'feature_importance': importance,
            'training_time': training_time,
            'evals_result': evals_result,
            'feature_info': train_info,
            'best_iteration': self.model.best_iteration
        }
    
    def _calculate_metrics(self, y_true, y_pred, dataset_name):
        """Calculate and print regression metrics."""
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        print(f"\n{dataset_name} Metrics:")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  MAE:  {mae:.4f}")
        print(f"  R²:   {r2:.4f}")
        
        return {
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'mse': mse
        }
    
    def save_model(self, model_path):
        """Save the trained model."""
        if self.model is None:
            raise ValueError("No model to save. Train a model first.")
        
        self.model.save_model(model_path)
        print(f"Model saved to {model_path}")
    
    def load_model(self, model_path):
        """Load a pre-trained model."""
        self.model = lgb.Booster(model_file=model_path)
        print(f"Model loaded from {model_path}")
    
    def predict(self, h5_file_path):
        """Make predictions on new data."""
        if self.model is None:
            raise ValueError("No model loaded. Train or load a model first.")
        
        X, _, _ = self.load_and_prepare_data(h5_file_path)
        predictions = self.model.predict(X, num_iteration=self.model.best_iteration)
        
        del X
        gc.collect()
        
        return predictions
    
    def analyze_feature_importance(self, top_n=20):
        """Analyze and display feature importance."""
        if self.model is None:
            raise ValueError("No model to analyze. Train a model first.")
        
        # Get feature importance
        importance = self.model.feature_importance(importance_type='gain')
        feature_names = [f'feat_{i}' for i in range(len(importance))]
        
        # Sort by importance
        sorted_indices = np.argsort(importance)[::-1]
        
        print(f"\nTop {top_n} Most Important Features:")
        print("-" * 40)
        for i in range(min(top_n, len(sorted_indices))):
            feature_idx = sorted_indices[i]
            feature_type = self._get_feature_type(feature_idx)
            print(f"{i+1:2d}. {feature_names[feature_idx]} ({feature_type}): {importance[feature_idx]:.2f}")
        
        return list(zip(feature_names, importance))
    
    def _get_feature_type(self, feature_idx):
        """Determine which feature type an index belongs to."""
        if feature_idx < 512:  # Assuming CLIP embedding dimension
            return "text_embed"
        elif feature_idx < 1024:
            return "img_embed"
        elif feature_idx < 1028:
            return "text_knn"
        else:
            return "img_knn"

In [ ]:
def incremental_train_lgbm(train_h5_files, val_h5_files, 
                          initial_params=None, save_path='lgbm_price_model.txt'):
    """
    Incrementally train LightGBM model on multiple training files.
    
    Args:
        train_h5_files: List of training H5 files
        val_h5_files: List of validation H5 files
        initial_params: Optional initial parameters
        save_path: Path to save the final model
    """
    print("=" * 60)
    print("INCREMENTAL LIGHTGBM TRAINING WITH REGULARIZATION")
    print("=" * 60)
    
    trainer = LGBMFeatureTrainer(use_gpu=True, gpu_id=0)
    
    # Load and combine all validation data first
    print("Loading and combining validation data...")
    all_X_val = []
    all_y_val = []
    
    for i, val_file in enumerate(val_h5_files):
        print(f"Loading validation file {val_file} ({i+1}/{len(val_h5_files)})...")
        X_val, y_val, _ = trainer.load_and_prepare_data(val_file)
        all_X_val.append(X_val)
        all_y_val.append(y_val)
        
        del X_val, y_val
        gc.collect()
    
    # Concatenate all validation data
    print("Concatenating validation data...")
    X_val_combined = np.concatenate(all_X_val, axis=0)
    y_val_combined = np.concatenate(all_y_val, axis=0)
    
    del all_X_val, all_y_val
    gc.collect()
    
    print(f"Combined validation data shape: {X_val_combined.shape}")
    print(f"Combined validation targets shape: {y_val_combined.shape}")
    
    # Initialize model with first training file
    print(f"\n{'='*50}")
    print(f"STAGE 1: INITIAL TRAINING ON {train_h5_files[0]}")
    print(f"{'='*50}")
    
    # Load first training file
    X_train_1, y_train_1, feature_info = trainer.load_and_prepare_data(train_h5_files[0])
    
    # Set up strong regularization parameters to prevent overfitting
    if initial_params is None:
        params = {
            'objective': 'regression',
            'metric': ['rmse', 'mae'],
            'boosting_type': 'gbdt',
            'learning_rate': 0.03,  # Lower learning rate
            'num_leaves': 31,
            'max_depth': 6,
            
            # Strong regularization
            'min_data_in_leaf': 25,
            'min_sum_hessian_in_leaf': 1e-3,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'lambda_l1': 0.2,  # Increased L1 regularization
            'lambda_l2': 0.2,  # Increased L2 regularization
            'min_gain_to_split': 0.02,
            
            # Performance
            'verbosity': 1,
            'seed': 42,
            'deterministic': True,
            'force_row_wise': True,
            
            # GPU
            'device': 'gpu',
            'gpu_platform_id': 0,
            'gpu_device_id': 0,
            'num_threads': -1,
        }
    else:
        params = initial_params.copy()
    
    print("Training Parameters:")
    for key, value in params.items():
        print(f"  {key}: {value}")
    
    # Create datasets for first stage - KEY FIX: free_raw_data=False
    train_data_1 = lgb.Dataset(
        X_train_1, 
        label=y_train_1,
        feature_name=[f'feat_{i}' for i in range(X_train_1.shape[1])],
        free_raw_data=False  # CRITICAL: Keep raw data for predictions
    )
    val_data = lgb.Dataset(
        X_val_combined, 
        label=y_val_combined, 
        reference=train_data_1,
        feature_name=[f'feat_{i}' for i in range(X_val_combined.shape[1])],
        free_raw_data=False  # CRITICAL: Keep raw data for predictions
    )
    
    # Train initial model
    evals_result = {}
    start_time = time.time()
    
    trainer.model = lgb.train(
        params=params,
        train_set=train_data_1,
        num_boost_round=1000,
        valid_sets=[train_data_1, val_data],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(50),
            lgb.record_evaluation(evals_result)
        ]
    )
    
    stage_1_time = time.time() - start_time
    print(f"Stage 1 completed in {stage_1_time:.2f} seconds")
    print(f"Best iteration: {trainer.model.best_iteration}")
    
    # Evaluate stage 1
    val_pred_1 = trainer.model.predict(X_val_combined, num_iteration=trainer.model.best_iteration)
    stage_1_metrics = trainer._calculate_metrics(y_val_combined, val_pred_1, "Stage 1 Validation")
    
    # Clean up stage 1 data (but keep validation data and arrays)
    del X_train_1, y_train_1, train_data_1
    gc.collect()
    
    # Continue training with remaining files
    all_metrics = [stage_1_metrics]
    
    for stage, train_file in enumerate(train_h5_files[1:], 2):
        print(f"\n{'='*50}")
        print(f"STAGE {stage}: INCREMENTAL TRAINING ON {train_file}")
        print(f"{'='*50}")
        
        # Load current training data
        X_train_curr, y_train_curr, _ = trainer.load_and_prepare_data(train_file)
        
        # Create new training dataset - KEY FIX: free_raw_data=False
        train_data_curr = lgb.Dataset(
            X_train_curr, 
            label=y_train_curr,
            feature_name=[f'feat_{i}' for i in range(X_train_curr.shape[1])],
            free_raw_data=False  # CRITICAL: Keep raw data for predictions
        )
        
        # Reduce learning rate for incremental training
        params_incremental = params.copy()
        params_incremental['learning_rate'] = max(0.01, params['learning_rate'] * 0.7)  # Reduce LR
        params_incremental['lambda_l1'] *= 1.2  # Increase regularization
        params_incremental['lambda_l2'] *= 1.2
        
        # print(f"Incremental learning rate: {params_incremental['learning_rate']:.4f}")
        
        # Continue training from existing model
        evals_result_stage = {}
        stage_start_time = time.time()
        
        trainer.model = lgb.train(
            params=params_incremental,
            train_set=train_data_curr,
            num_boost_round=500,  # Fewer rounds for incremental training
            valid_sets=[train_data_curr, val_data],
            valid_names=['train', 'val'],
            init_model=trainer.model,  # Continue from previous model
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(25),
                lgb.record_evaluation(evals_result_stage)
            ]
        )
        
        stage_time = time.time() - stage_start_time
        # print(f"Stage {stage} completed in {stage_time:.2f} seconds")
        print(f"Best iteration: {trainer.model.best_iteration}")
        
        # Evaluate current stage
        val_pred_curr = trainer.model.predict(X_val_combined, num_iteration=trainer.model.best_iteration)
        stage_metrics = trainer._calculate_metrics(y_val_combined, val_pred_curr, f"Stage {stage} Validation")
        all_metrics.append(stage_metrics)
        
        # Clean up current stage data
        del X_train_curr, y_train_curr, train_data_curr
        gc.collect()
    
    # Final evaluation
    print("\n" + "=" * 60)
    print("FINAL INCREMENTAL TRAINING RESULTS")
    print("=" * 60)
    
    final_val_pred = trainer.model.predict(X_val_combined, num_iteration=trainer.model.best_iteration)
    final_metrics = trainer._calculate_metrics(y_val_combined, final_val_pred, "Final Validation")
    
    # Compare metrics across stages
    print("\nMetrics progression across stages:")
    print("-" * 50)
    for i, metrics in enumerate(all_metrics, 1):
        print(f"Stage {i}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R²={metrics['r2']:.4f}")
    
    # Save final model
    trainer.save_model(save_path)
    
    # Feature importance analysis
    print("\nAnalyzing final feature importance...")
    importance = trainer.analyze_feature_importance(top_n=30)
    
    # Clean up
    del X_val_combined, y_val_combined, val_data
    gc.collect()
    
    return {
        'trainer': trainer,
        'final_metrics': final_metrics,
        'all_stage_metrics': all_metrics,
        'feature_importance': importance,
        'best_iteration': trainer.model.best_iteration
    }

In [8]:
def main_lgbm_incremental_training():
    """Main function to perform incremental LightGBM training."""
    
    # Define file paths
    train_files = [
        'train_features_1.h5',
        'train_features_2.h5',
        'train_features_3.h5'
    ]
    
    val_files = [
        'val_features_1.h5',
        'val_features_2.h5',
        'val_features_3.h5'
    ]
    
    print("Starting LightGBM incremental training...")
    print(f"Training files: {train_files}")
    print(f"Validation files: {val_files}")
    
    try:
        # Run incremental training
        results = incremental_train_lgbm(
            train_h5_files=train_files,
            val_h5_files=val_files,
            save_path='lgbm_price_model_incremental.txt'
        )
        
        print("\n" + "=" * 60)
        print("TRAINING COMPLETED SUCCESSFULLY!")
        print("=" * 60)
        
        # Print final summary
        final_metrics = results['final_metrics']
        print(f"\nFinal Model Performance:")
        print(f"  RMSE: {final_metrics['rmse']:.4f}")
        print(f"  MAE:  {final_metrics['mae']:.4f}")
        print(f"  R²:   {final_metrics['r2']:.4f}")
        print(f"  Best Iteration: {results['best_iteration']}")
        
        # Check for overfitting by comparing first and last stage
        first_stage = results['all_stage_metrics'][0]
        last_stage = results['all_stage_metrics'][-1]
        
        print(f"\nOverfitting Analysis:")
        print(f"  Stage 1 RMSE: {first_stage['rmse']:.4f}")
        print(f"  Final RMSE:   {last_stage['rmse']:.4f}")
        print(f"  Improvement:  {(first_stage['rmse'] - last_stage['rmse']):.4f}")
        
        if last_stage['rmse'] < first_stage['rmse']:
            print("  ✓ Model improved with additional data!")
        else:
            print("  ⚠ Model performance degraded - consider stronger regularization")
        
        return results
        
    except Exception as e:
        print(f"Training failed with error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Execute the incremental training
print("Starting LightGBM incremental training to combat overfitting...")
lgbm_results = main_lgbm_incremental_training()

Starting LightGBM incremental training to combat overfitting...
Starting LightGBM incremental training...
Training files: ['train_features_1.h5', 'train_features_2.h5', 'train_features_3.h5']
Validation files: ['val_features_1.h5', 'val_features_2.h5', 'val_features_3.h5']
INCREMENTAL LIGHTGBM TRAINING WITH REGULARIZATION
Loading and combining validation data...
Loading validation file val_features_1.h5 (1/3)...
Loading data from val_features_1.h5...
Loaded feature shapes:
  text_embeds: (4980, 512)
  img_embeds: (4980, 512)
  text_knn: (4980, 4)
  img_knn: (4980, 4)
  prices: (4980,)
Combined feature matrix shape: (4980, 1032)
Target shape: (4980,)
Loading validation file val_features_2.h5 (2/3)...
Loading data from val_features_2.h5...
Loaded feature shapes:
  text_embeds: (4981, 512)
  img_embeds: (4981, 512)
  text_knn: (4981, 4)
  img_knn: (4981, 4)
  prices: (4981,)
Combined feature matrix shape: (4981, 1032)
Target shape: (4981,)
Loading validation file val_features_2.h5 (2/3)..

In [10]:
import pickle 
lgbm_results['trainer'].save_model('lgbm_price_model_final.txt')

# Also save model metadata and results

model_metadata = {
    'final_metrics': lgbm_results['final_metrics'],
    'all_stage_metrics': lgbm_results['all_stage_metrics'],
    'best_iteration': lgbm_results['best_iteration'],
    'feature_importance': lgbm_results['feature_importance']
}

with open('lgbm_model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print("LightGBM model and metadata saved successfully!")
print(f"Model saved to: lgbm_price_model_final.txt")
print(f"Metadata saved to: lgbm_model_metadata.pkl")# Save the trained LightGBM model
lgbm_results['trainer'].save_model('lgbm_price_model_final.txt')

# Also save model metadata and results

model_metadata = {
    'final_metrics': lgbm_results['final_metrics'],
    'all_stage_metrics': lgbm_results['all_stage_metrics'],
    'best_iteration': lgbm_results['best_iteration'],
    'feature_importance': lgbm_results['feature_importance']
}

with open('lgbm_model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print("LightGBM model and metadata saved successfully!")
print(f"Model saved to: lgbm_price_model_final.txt")
print(f"Metadata saved to: lgbm_model_metadata.pkl")# Save the trained LightGBM model
lgbm_results['trainer'].save_model('lgbm_price_model_final.txt')

# Also save model metadata and results

model_metadata = {
    'final_metrics': lgbm_results['final_metrics'],
    'all_stage_metrics': lgbm_results['all_stage_metrics'],
    'best_iteration': lgbm_results['best_iteration'],
    'feature_importance': lgbm_results['feature_importance']
}

with open('lgbm_model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print("LightGBM model and metadata saved successfully!")
print(f"Model saved to: lgbm_price_model_final.txt")
print(f"Metadata saved to: lgbm_model_metadata.pkl")

Model saved to lgbm_price_model_final.txt
LightGBM model and metadata saved successfully!
Model saved to: lgbm_price_model_final.txt
Metadata saved to: lgbm_model_metadata.pkl
Model saved to lgbm_price_model_final.txt
LightGBM model and metadata saved successfully!
Model saved to: lgbm_price_model_final.txt
Metadata saved to: lgbm_model_metadata.pkl
Model saved to lgbm_price_model_final.txt
LightGBM model and metadata saved successfully!
Model saved to: lgbm_price_model_final.txt
Metadata saved to: lgbm_model_metadata.pkl
